In [ ]:
import numpy as np
import sympy

import qualtran as qlt
import qualtran.dtype as qdt
from qualtran.l3 import bloqify

In [ ]:
from qualtran.l3.tests.recursive_and_test import AndR, SymbolicMultiAnd

In [ ]:
@bloqify
def bfunc(bb, ctrl):
    out = bb.alloc_qbit()
    ctrl, out = bb.CNOT(ctrl, out)
    return {'ctrl': ctrl, 'out': out}
    

In [ ]:
bbloq = bfunc.make(qlt.Signature.build(ctrl=1, out=(None, qdt.QBit())))
qlt.show_bloq(bbloq, 'musical_score')

In [ ]:
n = 3
multiand = AndR.make(qlt.Signature.build(ctrl=1, x=qdt.QAny(n)))

In [ ]:
qlt.show_bloq(multiand)

In [ ]:
qlt.show_bloq(multiand.flatten_once())

In [ ]:
qlt.show_bloq(multiand.flatten())

## Symbolic

In [ ]:
from sympy import Function, rsolve, symbols
n = symbols('n', integer=True, positive=True)
L = Function('L')
                 
sma = SymbolicMultiAnd(n)
qlt.show_bloq(sma)

In [ ]:
qlt.show_bloq(sma.decompose_bloq())

In [ ]:



# # 2. Define the recurrence relation

# recurrence_relation = L(n+1) - L(n) - 2

# # 3. Define the initial condition
# # L(0) = 2
# initial_condition = {L(1): 2}

# # 4. Solve using rsolve
# result = rsolve(recurrence_relation, L(n), initial_condition)

# print(f"Closed form cost: {result}")
# result

In [ ]:
import logging
from typing import Callable, Set

import networkx as nx
from attrs import frozen

from qualtran import (
    Bloq,
    CompositeBloq,
    Connection,
    DanglingT,
    DecomposeNotImplementedError,
    DecomposeTypeError,
)
from qualtran._infra.binst_graph_iterators import greedy_topological_sort
from qualtran._infra.composite_bloq import _binst_to_cxns
from qualtran.symbolics import smax, SymbolicInt

logger = logging.getLogger(__name__)


binst_graph = sma.decompose_bloq()._binst_graph
def _bloq_max_width(bloq):
    if isinstance(bloq, SymbolicMultiAnd):
        return L(bloq.n)
    return bloq.signature.n_bits()

max_width: SymbolicInt = 0
in_play: Set[Connection] = set()

for cc in nx.weakly_connected_components(binst_graph):
    for binst in greedy_topological_sort(binst_graph.subgraph(cc)):
        pred_cxns, succ_cxns = _binst_to_cxns(binst, binst_graph=binst_graph)

        # Remove inbound connections from those that are 'in play'.
        for cxn in pred_cxns:
            in_play.remove(cxn)

        if not isinstance(binst, DanglingT):
            # During the application of the binst, we have "observer" connections that have
            # width as well as the width from the binst itself. We consider the case where
            # the bloq may have a max_width greater than the max of its left/right registers.
            during_size = _bloq_max_width(binst.bloq) + sum(s.shape for s in in_play)
            max_width = smax(max_width, during_size)

        # After the binst, its successor connections are 'in play'.
        in_play.update(succ_cxns)
        after_size = sum(s.shape for s in in_play)
        max_width = smax(max_width, after_size)

max_width

In [ ]:
# Note our base case is L(1)
# so we need n-1 >= 1
# So L(n) is called with n = 2 to use the base case.
# max(0, 2 + 3, L(0) + 2) = max(0, 5, 4)
# but anything other than that will simplify to L(n-1) + 2
max_width = L(n-1) + 2

In [ ]:
recurrence_relation = L(n) - max_width

# 3. Define the initial condition
# L(0) = 2
initial_condition = {L(1): 2}

# 4. Solve using rsolve
result = rsolve(recurrence_relation, L(n), initial_condition)
result